# Approach 1 — Statistical Pre-Screener Only

**Thesis Section: Chapter 4.1**

This notebook runs the statistical-only baseline (Approach 1) on all 55 industrial signals.
No LLM is used — the pre-screener alone classifies each signal.

Pre-screener rules:
- Frozen: rolling 10-pt std < 0.01
- Drift: rolling 96-pt mean range > 15% of signal range (with seasonality filter)
- Spikes: local 4σ deviation with shift(5) pre-spike baseline
- Intermittent: negative values < 5% of signal

In [ ]:
import sys
sys.path.insert(0, '../..')

from dotenv import load_dotenv
load_dotenv('../../.env')

import numpy as np
import pandas as pd
from src.data.timeseer_client import fetch_series_api, list_series_api, AREAS
from src.data.ground_truth import GROUND_TRUTH, AREA_TAGS, LABEL_NAMES
from src.prescreener.analyze import hybrid_analyze, SKIP_SPIKE_TAGS
from src.evaluation.metrics import compute_metrics
from src.evaluation.report import print_summary_table, save_results

print('Imports OK')

In [ ]:
# Run statistical pre-screener on all signals
# (Uses Approach 3 pre-screener but with final detected[] as the prediction)

all_results = []

for area in AREAS.keys():
    print(f'\n=== {area} ===')
    tags = list_series_api(area)
    
    for tag in tags:
        vals, idx = fetch_series_api(tag, area)
        if vals is None:
            continue
        
        # Pre-screener only — use detected[] as prediction
        _, _, _, detected, _ = hybrid_analyze(vals, idx, tag)[2:]
        
        # Map detected flags to category letter
        pred_map = {
            'stale': 'C', 'drift': 'A', 'spikes': 'B',
            'var_collapse': 'G', 'intermittent': 'L', 'clean': 'E'
        }
        if len(detected) == 2 and 'intermittent' in detected and 'spikes' in detected:
            pred = 'B+L'
        else:
            pred = pred_map.get(detected[0], 'E')
        
        gt = GROUND_TRUTH.get(tag, '?')
        print(f'  {tag:<30} detected={detected}  pred={pred}  gt={gt}')
        all_results.append({'Tag': tag, 'Category': pred, 'gt': gt,
                            'Detected': ', '.join(detected)})

print(f'\nTotal: {len(all_results)} signals')

In [ ]:
# Evaluate
preds = [r['Category'] for r in all_results if r['gt'] != '?']
gts   = [r['gt']      for r in all_results if r['gt'] != '?']

metrics = compute_metrics(preds, gts)
print(f'Approach 1 (Statistical Only)')
print(f'  Accuracy  : {metrics["accuracy"]:.3f}  ({metrics["n_correct"]}/{metrics["n_samples"]})')
print(f'  Precision : {metrics["precision"]:.3f}')
print(f'  Recall    : {metrics["recall"]:.3f}')
print(f'  F1        : {metrics["f1"]:.3f}')
print()
print(metrics['report'])